# Re-run: agreement with the human raters (RQ2)

This notebook recomputes the point estimates of Table 2 (each format's correlation with the human mean,
the human ceiling, and the difference dH) and of Table 3 (signed error by band of human score) from the
per-case file published with release `checklist-grading-2026`. It needs Python 3.9+ with numpy and
`per-case-model-outputs.jsonl` beside this notebook (or in `RELEASE_DIR`). The 95% intervals come from
the release code package's 4,000-replicate cluster bootstrap; this notebook checks the point estimates.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "checklist-grading-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import json
import numpy as np

recs = [json.loads(l) for l in open(find_release_file(), encoding="utf-8")]
G = {(r["corpus"], r["id"], r["form"], r["model"], r["arm"], r["rep"]): r for r in recs if r["record"] == "grade"}
items = {(r["corpus"], r["id"]): r for r in recs if r["record"] == "item"}
MODELS, ARMS, REPS = ("gemini", "gpt"), ("likert", "boolean"), (0, 1)
DIMS = ["Correctness", "Code Elegance", "Readability", "Documentation"]
r = lambda x, y: float(np.corrcoef(x, y)[0, 1])

def val(corpus, i, m, a, k, dim=None):
    g = G.get((corpus, i, "orig", m, a, k))
    return None if g is None else (g["score"] if dim is None else (g["scores"] or {}).get(dim))

def complete(corpus, ids, dim=None):
    return [i for i in ids if all(val(corpus, i, m, a, k, dim) is not None for m in MODELS for a in ARMS for k in REPS)]

def mean_score(corpus, i, m, a, dim=None):
    return (val(corpus, i, m, a, 0, dim) + val(corpus, i, m, a, 1, dim)) / 2

In [3]:
mohler = complete("mohler", sorted(i for (c, i), it in items.items() if c == "mohler" and "main" in it["draws"]))
h = np.array([items["mohler", i]["h"] for i in mohler])
V = {(m, a): r(np.array([mean_score("mohler", i, m, a) for i in mohler]), h) for m in MODELS for a in ARMS}
dV = np.mean([V[m, "boolean"] - V[m, "likert"] for m in MODELS])
ceiling = r([items["mohler", i]["g1"] for i in mohler], [items["mohler", i]["g2"] for i in mohler])
print(f"Mohler n={len(mohler)}: gemini {V['gemini','likert']:.3f}/{V['gemini','boolean']:.3f}, "
      f"gpt {V['gpt','likert']:.3f}/{V['gpt','boolean']:.3f}, dH {dV:+.3f}, raters {ceiling:.3f}")
assert (f"{V['gemini','likert']:.3f}", f"{V['gemini','boolean']:.3f}", f"{V['gpt','likert']:.3f}",
        f"{V['gpt','boolean']:.3f}", f"{dV:+.3f}", f"{ceiling:.3f}") == ("0.764", "0.688", "0.819", "0.682", "-0.106", "0.621")

Mohler n=620: gemini 0.764/0.688, gpt 0.819/0.682, dH -0.106, raters 0.621


In [4]:
men = complete("menagerie", sorted((i for (c, i) in items if c == "menagerie"), key=int), DIMS[0])
PRINTED = {"Correctness": ("0.405", "0.433", "-0.173", "-0.054", "+0.073", "0.622"),
           "Code Elegance": ("0.035", "0.017", "-0.182", "-0.137", "+0.013", "0.539"),
           "Readability": ("0.039", "-0.007", "-0.235", "-0.011", "+0.089", "0.571"),
           "Documentation": ("0.286", "0.079", "-0.284", "-0.125", "-0.024", "0.525")}
for d in DIMS:
    hm = np.array([items["menagerie", i]["h"][d] for i in men])
    Vd = {(m, a): r(np.array([mean_score("menagerie", i, m, a, d) for i in men]), hm) for m in MODELS for a in ARMS}
    dVd = np.mean([Vd[m, "boolean"] - Vd[m, "likert"] for m in MODELS])
    one, rest = [], []                      # one rater against the mean of the others, pooled over ratings
    for i in men:
        g = items["menagerie", i]["raters"][d]
        for j in range(len(g)):
            one.append(g[j]); rest.append(np.mean(g[:j] + g[j + 1:]))
    got = (f"{Vd['gemini','likert']:.3f}", f"{Vd['gemini','boolean']:.3f}", f"{Vd['gpt','likert']:.3f}",
           f"{Vd['gpt','boolean']:.3f}", f"{dVd:+.3f}", f"{r(one, rest):.3f}")
    print(f"Menagerie {d:14s} n={len(men)}: {got}")
    assert got == PRINTED[d], (d, got)
print("Table 2's point estimates reproduce.")

Menagerie Correctness    n=226: ('0.405', '0.433', '-0.173', '-0.054', '+0.073', '0.622')
Menagerie Code Elegance  n=226: ('0.035', '0.017', '-0.182', '-0.137', '+0.013', '0.539')
Menagerie Readability    n=226: ('0.039', '-0.007', '-0.235', '-0.011', '+0.089', '0.571')
Menagerie Documentation  n=226: ('0.286', '0.079', '-0.284', '-0.125', '-0.024', '0.525')
Table 2's point estimates reproduce.


In [5]:
# Table 3: signed error (score minus human mean / 5), pooled over models and calls, by band of human score.
eL = np.array([np.mean([val("mohler", i, m, "likert", k) for m in MODELS for k in REPS]) for i in mohler]) - h / 5
eB = np.array([np.mean([val("mohler", i, m, "boolean", k) for m in MODELS for k in REPS]) for i in mohler]) - h / 5
BANDS = {"LOW": h <= 2.5, "MID": (h > 2.5) & (h < 4.5), "HIGH": h >= 4.5}
PRINTED3 = {"LOW": (65, "-0.144", "-0.123", "+0.022"), "MID": (145, "-0.149", "-0.327", "-0.178"),
            "HIGH": (410, "-0.049", "-0.160", "-0.111")}
for b, mask in BANDS.items():
    got = (int(mask.sum()), f"{eL[mask].mean():+.3f}", f"{eB[mask].mean():+.3f}", f"{eB[mask].mean() - eL[mask].mean():+.3f}")
    print(b, got)
    assert got == PRINTED3[b], (b, got)
print("Table 3's point estimates reproduce.")

LOW (65, '-0.144', '-0.123', '+0.022')
MID (145, '-0.149', '-0.327', '-0.178')
HIGH (410, '-0.049', '-0.160', '-0.111')
Table 3's point estimates reproduce.
